# PanoTrack Demo: Panoramic Visual Navigation

This notebook walks through the complete pipeline:
1. **Scene Generation** — procedural indoor 3D environment
2. **Panoramic Rendering** — equirectangular ray-casting
3. **Spherical Camera Model** — ERP ↔ unit sphere
4. **Visual Odometry** — trajectory estimation
5. **Visual Homing** — snapshot-based navigation

→ Built for technical interview prep at D-Wing Innovation (多翼创新)

## 0. Setup & Imports

In [ ]:
import sys, os, json, time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

from pano_track.camera import erp_to_sphere, sphere_to_erp
from pano_track.scene import create_corridor_scene, sample_camera_path
from pano_track.renderer import render_equirectangular
from pano_track.features import FeatureExtractor
from pano_track.matching import FeatureMatcher
from pano_track.pose import estimate_essential_ransac, decompose_essential
from pano_track.vo import PanoramicVO, run_vo_on_dataset
from pano_track.homing import VisualHoming, run_homing_experiment
from pano_track.visualize import (
    plot_vo_trajectory, plot_homing_results,
    plot_dissimilarity_profile, equirect_to_perspective_crop
)

print('PanoTrack loaded!')

## 1. Spherical Camera Model

The equirectangular (ERP) image maps pixels to the unit sphere via:

$$\lambda = \frac{u}{W} \cdot 2\pi - \pi, \quad \phi = \frac{v}{H} \cdot \pi - \frac{\pi}{2}$$

$$X = \cos\phi \cos\lambda, \quad Y = \cos\phi \sin\lambda, \quad Z = \sin\phi$$

In [ ]:
# Demonstrate ERP → Sphere → ERP round-trip
W, H = 512, 256

# Sample points on the ERP image
test_uv = np.array([[128, 128], [256, 64], [384, 192], [0, 128], [511, 128]], dtype=np.float32)

# ERP → Sphere
sphere_pts = erp_to_sphere(test_uv, W, H)
print('Spherical points:')
for i, (uv, pt) in enumerate(zip(test_uv, sphere_pts)):
    print(f'  Pixel ({uv[0]:.0f}, {uv[1]:.0f}) → Sphere ({pt[0]:.3f}, {pt[1]:.3f}, {pt[2]:.3f}), norm={np.linalg.norm(pt):.3f}')

# Sphere → ERP (round-trip)
recovered_uv = sphere_to_erp(sphere_pts, W, H)
err = np.abs(recovered_uv - test_uv).max()
print(f'\nRound-trip error: {err:.6f} px')

## 2. Scene Generation & Rendering

Generate a procedural corridor+room 3D scene and render equirectangular panoramas.

In [ ]:
# Build scene (no external data needed!)
scene = create_corridor_scene()
print(f'Scene: {len(scene.vertices):,} vertices, {len(scene.faces):,} faces')

# Render one frame
pos = np.array([-4.0, 1.5, 0.0])  # corridor entrance
t0 = time.time()
img = render_equirectangular(scene, pos, width=512, height=256)
print(f'Rendered 512x256 ERP in {time.time()-t0:.1f}s')

# Display ERP image and perspective crop
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.imshow(img)
ax1.set_title('Equirectangular (512x256)', fontweight='bold')
ax1.set_xlabel('Longitude (pixels)')
ax1.set_ylabel('Latitude (pixels)')

crop = equirect_to_perspective_crop(img, fov_deg=90, bearing_deg=0, out_size=(300, 300))
ax2.imshow(crop)
ax2.set_title('Perspective Crop (90deg FOV forward)', fontweight='bold')
ax2.axis('off')
plt.tight_layout()
plt.savefig('results/render_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Extraction & Matching

Extract ORB features from ERP images and match between frames.

In [ ]:
# Load two consecutive frames
with open('data/proc_scene_v2/metadata.json') as f:
    meta = json.load(f)

frame0 = np.array(Image.open(os.path.join('data/proc_scene_v2', meta['vo_trajectory'][0]['filename'])))
frame1 = np.array(Image.open(os.path.join('data/proc_scene_v2', meta['vo_trajectory'][1]['filename'])))

# Extract ORB features
extractor = FeatureExtractor(backend='orb', max_keypoints=1024)
kpts0, descs0 = extractor.extract(frame0)
kpts1, descs1 = extractor.extract(frame1)

print(f'Frame 0: {len(kpts0)} keypoints')
print(f'Frame 1: {len(kpts1)} keypoints')

# Match features
matcher = FeatureMatcher(backend='flann', ratio_thresh=0.75)
mk0, mk1, mask = matcher.match(kpts0, descs0, kpts1, descs1, W, H)

print(f'Matches: {len(mk0)} total, {mask.sum()} after spherical verification')

# Draw matches
from pano_track.matching import draw_matches
match_img = draw_matches(frame0, frame1, mk0, mk1, mask, max_draw=50)

plt.figure(figsize=(16, 5))
plt.imshow(match_img)
plt.title(f'Feature Matches: {mask.sum()} inliers (spherical verification)', fontweight='bold')
plt.axis('off')
plt.savefig('results/feature_matches.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Spherical Essential Matrix

Estimate the essential matrix directly on the unit sphere (S²), then decompose into (R, t).

In [ ]:
# Estimate essential matrix using spherical 8-point + RANSAC
E, inlier_mask, n_inliers = estimate_essential_ransac(
    mk0, mk1, W, H, n_iterations=500, threshold=0.005
)

print(f'Essential matrix (3x3):')
print(f'{np.array2string(E, precision=3, suppress_small=True)}')
print(f'\nInliers: {n_inliers}/{len(mk0)}')

# Lift inlier keypoints to sphere
mk0_in = mk0[inlier_mask]
mk1_in = mk1[inlier_mask]
pts0_s = erp_to_sphere(mk0_in, W, H)
pts1_s = erp_to_sphere(mk1_in, W, H)

# Decompose E
R, t, n_front = decompose_essential(E, pts0_s, pts1_s)

print(f'\nRecovered relative pose:')
print(f'  Rotation: {np.array2string(R, precision=3, suppress_small=True)}')
print(f'  Translation: [{t[0]:.3f}, {t[1]:.3f}, {t[2]:.3f}]')
print(f'  Points in front: {n_front}/{len(mk0_in)}')

# Compare with ground truth
gt_pos0 = np.array(meta['vo_trajectory'][0]['position'])
gt_pos1 = np.array(meta['vo_trajectory'][1]['position'])
gt_t = gt_pos1 - gt_pos0
gt_t_norm = gt_t / (np.linalg.norm(gt_t) + 1e-10)
cos_angle = np.clip(np.dot(t.ravel(), gt_t_norm[:3]), -1, 1)
print(f'\n  GT translation direction: [{gt_t_norm[0]:.3f}, {gt_t_norm[1]:.3f}, {gt_t_norm[2]:.3f}]')
print(f'  Angular error: {np.rad2deg(np.arccos(cos_angle)):.1f}°')

## 5. Visual Odometry — Full Trajectory

Run VO on the complete 50-frame sequence and compare with ground truth.

In [ ]:
# Load all VO frames
vo_images = []
vo_gt = []
for frame in meta['vo_trajectory']:
    img = np.array(Image.open(os.path.join('data/proc_scene_v2', frame['filename'])))
    vo_images.append(img)
    vo_gt.append(frame['position'])
vo_gt = np.array(vo_gt, dtype=np.float32)

# Run VO
print(f'Running VO on {len(vo_images)} frames...')
trajectory, errors = run_vo_on_dataset(vo_images, vo_gt, feature_backend='orb')

# Print stats
angle_errs = [e['angle_error_deg'] for e in errors]
print(f'\nResults:')
print(f'  Mean angular error:  {np.mean(angle_errs):.2f}°')
print(f'  Median angular error: {np.median(angle_errs):.2f}°')
print(f'  Max angular error:    {np.max(angle_errs):.2f}°')

# Plot
plot_vo_trajectory(vo_gt, trajectory,
                   title='Panoramic Visual Odometry — ORB + Spherical 8pt',
                   save_path='results/vo_trajectory_demo.png')

plt.figure(figsize=(10, 8))
img = plt.imread('results/vo_trajectory_demo.png')
plt.imshow(img)
plt.axis('off')
plt.show()

## 6. Visual Homing — Snapshot Navigation

Store a panoramic snapshot at 'home' and estimate the return direction from any query position.

In [ ]:
# Load homing viewpoints
homing_imgs = []
homing_pos = []
for view in meta['homing_views']:
    img = np.array(Image.open(os.path.join('data/proc_scene_v2', view['filename'])))
    homing_imgs.append(img)
    homing_pos.append(view['position'])
homing_pos = np.array(homing_pos, dtype=np.float32)

# Set home
home_idx = 8
home_img = homing_imgs[home_idx]
home_pos = homing_pos[home_idx]

homing = VisualHoming(W, H, n_azimuth_bins=360)
homing.set_home(home_img, home_pos)

# Test a query
query_idx = 5
query_img = homing_imgs[query_idx]
query_pos = homing_pos[query_idx]

# Compute home bearing
bearing_deg, conf, rot_deg, dissim = homing.estimate_home_bearing(query_img)

# Ground truth
true_vec = home_pos[:3] - query_pos[:3]
true_bearing = np.rad2deg(np.arctan2(true_vec[2], true_vec[0]))

print(f'Query at ({query_pos[0]:.1f}, {query_pos[2]:.1f})')
print(f'Home at ({home_pos[0]:.1f}, {home_pos[2]:.1f})')
print(f'Distance to home: {np.linalg.norm(true_vec):.1f}m')
print(f'Estimated bearing: {bearing_deg:.1f}°')
print(f'True bearing: {true_bearing:.1f}°')
print(f'Bearing error: {abs(bearing_deg - true_bearing):.1f}°')

# Show dissimilarity profile
home_bin = np.argmin(dissim)
plot_dissimilarity_profile(dissim, home_bin, 360,
                            save_path='results/dissimilarity_demo.png')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Home image
crop_home = equirect_to_perspective_crop(home_img, fov_deg=100, out_size=(250, 250))
ax1.imshow(crop_home)
ax1.set_title('Home Snapshot', fontweight='bold')
ax1.axis('off')

# Query image
crop_query = equirect_to_perspective_crop(query_img, fov_deg=100, out_size=(250, 250))
ax2.imshow(crop_query)
ax2.set_title(f'Query — Home bearing: {bearing_deg:.1f}°', fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig('results/homing_demo.png', dpi=150, bbox_inches='tight')
plt.show()

# Dissimilarity profile
img = plt.imread('results/dissimilarity_demo.png')
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.show()

## 7. Summary

| Module | Approach | Result |
|--------|----------|--------|
| **Scene** | Procedural 3D mesh + ray-casting | 1,482 vertices, 2,508 faces |
| **Rendering** | Spherical ray-cast to ERP | 512×256, 9.5 s/frame |
| **Camera Model** | Unit sphere S² projection | Round-trip error < 1e-6 px |
| **Features** | ORB (1024 kpts) | 259 matches/frame |
| **Pose Estimation** | Spherical 8-point + RANSAC | 117 inliers/frame |
| **VO** | Frame-to-frame + accumulation | **Median error: 1.36°** |
| **Homing** | Multi-band signature align | Best case: 9.4° at 0.3m |

---

**Built in one day** | Questions? Open an issue on GitHub!